In [2]:
import sys
sys.path.append('..')

from auto_pricer.car import Car

train, val, test = Car.from_hub("ShahidHKhan/cars_full_processed")
print(f"Loaded {len(train):,} training cars")
print(train[0])
print(train[0].summary)

Loaded 231,947 training cars
title='2013 Honda CR-V' make='Honda' model='CR-V' trim='EX-L AWD' year=2013 mileage=73111.0 price=17995.0 horsepower=185.0 body_type='SUV / Crossover' fuel_type='Gasoline' transmission='5-Speed Automatic' has_accidents=False frame_damaged=False full=None summary='Title: 2013 Honda CR-V EX-L AWD\nCategory: SUV\nMake: Honda\nDescription: Well-maintained, accident-free SUV with desirable features.\nDetails: 73,111 miles, gasoline, 5-speed automatic transmission.' prompt=None completion=None id=None
Title: 2013 Honda CR-V EX-L AWD
Category: SUV
Make: Honda
Description: Well-maintained, accident-free SUV with desirable features.
Details: 73,111 miles, gasoline, 5-speed automatic transmission.


In [3]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

vector = encoder.encode([train[0].summary])[0]
print(f"Vector shape: {vector.shape}")
print(vector[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector shape: (384,)
[ 0.00453662  0.03135079  0.02285563 -0.05667333  0.07892009  0.07955173
 -0.01930789  0.03260751  0.01954967 -0.0017893 ]


In [4]:
import chromadb
from tqdm import tqdm

DB = "cars_vectorstore"
client = chromadb.PersistentClient(path=DB)

collection_name = "cars"
existing = [c.name for c in client.list_collections()]

if collection_name not in existing:
    collection = client.create_collection(collection_name)
    BATCH_SIZE = 1000

    for i in tqdm(range(0, len(train), BATCH_SIZE)):
        batch = train[i:i + BATCH_SIZE]
        documents = [car.summary for car in batch]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"make": car.make, "year": car.year, "price": car.price} for car in batch]
        ids = [f"car_{i + j}" for j in range(len(batch))]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)
    print(f"Populated vectorstore with {len(train):,} cars")

collection = client.get_or_create_collection(collection_name)
print(f"Collection has {collection.count():,} entries")

Collection has 231,947 entries


In [5]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from collections import Counter

MAXIMUM_DATAPOINTS = 5_000

result = collection.get(
    include=['embeddings', 'documents', 'metadatas'],
    limit=MAXIMUM_DATAPOINTS
)
vectors = np.array(result['embeddings'])
documents = result['documents']
makes = [m['make'] for m in result['metadatas']]

tsne = TSNE(n_components=3, random_state=42, n_jobs=-1)
reduced = tsne.fit_transform(vectors)

top_makes = [m for m, _ in Counter(makes).most_common(8)]
COLORS = ['red','blue','green','orange','purple','cyan','brown','yellow']
colors = [COLORS[top_makes.index(m)] if m in top_makes else 'gray' for m in makes]

fig = go.Figure(data=[go.Scatter3d(
    x=reduced[:,0], y=reduced[:,1], z=reduced[:,2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Make: {m}<br>{d[:60]}..." for m, d in zip(makes, documents)],
    hoverinfo='text'
)])
fig.update_layout(title="Car Listings Vector Space (colored by make)", height=600)
fig.show()

In [6]:
def find_similars(car, n_results=5):
    vector = encoder.encode([car.summary]).astype(float).tolist()
    results = collection.query(
        query_embeddings=vector,
        n_results=n_results,
        include=['documents', 'metadatas']
    )
    documents = results['documents'][0]
    prices = [m['price'] for m in results['metadatas'][0]]
    return documents, prices

# Spot check
docs, prices = find_similars(test[0])
print(f"Query car: {test[0].summary}")
print(f"Actual price: ${test[0].price:,.0f}\n")
for doc, price in zip(docs, prices):
    print(f"${price:,.0f} — {doc[:80]}")

Query car: Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: A well-maintained, one-owner rental vehicle with a clean history, ready for its next driver.
Details: 30,443 miles, gasoline, automatic transmission.
Actual price: $10,990

$7,200 — Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description:
$8,991 — Title: 2015 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description:
$9,975 — Title: 2018 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description:
$13,950 — Title: 2019 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description:
$10,995 — Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description:


In [7]:
def make_context(similar_docs, prices):
    context = "Here are some similar used car listings with their prices:\n\n"
    for doc, price in zip(similar_docs, prices):
        context += f"Price: ${price:,.0f}\n{doc}\n\n"
    return context.strip()

print(make_context(docs, prices))

Here are some similar used car listings with their prices:

Price: $7,200
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained, one-owner vehicle with no reported accidents.
Details: 66,105 miles, gasoline-powered, automatic transmission.

Price: $8,991
Title: 2015 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained, one-owner vehicle with a clean history and below-market mileage.
Details: 56,966 miles, Gasoline, 6-Speed Automatic.

Price: $9,975
Title: 2018 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: This well-maintained former rental vehicle is a great option for budget-conscious buyers.
Details: 43,300 miles, gasoline, automatic transmission.

Price: $13,950
Title: 2019 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained and inspected pre-owned vehicle with a clean history.
Details: 15,324 miles, gasoline, automatic transmission.

Price: $10,995
Titl

In [8]:
from litellm import completion

def messages_for_rag(car):
    docs, prices = find_similars(car)
    context = make_context(docs, prices)
    message = f"""Estimate the price of this used car. Respond with the price only, no explanation.

Car to price:
{car.summary}

{context}"""
    return [{"role": "user", "content": message}]

def gemini_rag(car):
    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=messages_for_rag(car)
    )
    return response.choices[0].message.content

# Single test call first — don't scale up yet
print(messages_for_rag(test[0])[0]['content'])
print("---")
print(gemini_rag(test[0]))

Estimate the price of this used car. Respond with the price only, no explanation.

Car to price:
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: A well-maintained, one-owner rental vehicle with a clean history, ready for its next driver.
Details: 30,443 miles, gasoline, automatic transmission.

Here are some similar used car listings with their prices:

Price: $7,200
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained, one-owner vehicle with no reported accidents.
Details: 66,105 miles, gasoline-powered, automatic transmission.

Price: $8,991
Title: 2015 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained, one-owner vehicle with a clean history and below-market mileage.
Details: 56,966 miles, Gasoline, 6-Speed Automatic.

Price: $9,975
Title: 2018 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: This well-maintained former rental vehicle is a great option f

In [9]:
import modal

AutoPricer = modal.Cls.from_name("auto-pricer-service", "AutoPricer")
pricer = AutoPricer()

docs, prices = find_similars(test[0])
context = make_context(docs, prices)

result_with_rag = pricer.price.remote(test[0].summary, context)
result_without_rag = pricer.price.remote(test[0].summary)

print(f"Actual price: ${test[0].price:,.0f}")
print(f"Fine-tuned WITHOUT RAG: ${result_without_rag:,.0f}")
print(f"Fine-tuned WITH RAG: ${result_with_rag:,.0f}")
print(f"(Gemini + RAG was: $11,075)")

Actual price: $10,990
Fine-tuned WITHOUT RAG: $9,995
Fine-tuned WITH RAG: $9,995
(Gemini + RAG was: $11,075)


In [10]:
fake_context = """Here are some similar used car listings with their prices:

Price: $32,000
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained hatchback with a clean history.
Details: 69,487 miles, gasoline, automatic transmission.

Price: $29,500
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained hatchback with a clean history.
Details: 69,487 miles, gasoline, automatic transmission.

Price: $31,200
Title: 2017 Ford Fiesta SE Hatchback
Category: Hatchback
Make: Ford
Description: Well-maintained hatchback with a clean history.
Details: 69,487 miles, gasoline, automatic transmission."""

result_with_fake_context = pricer.price.remote(test[0].summary, fake_context)
print(f"Fine-tuned WITH extreme fake context ($30k comps): ${result_with_fake_context:,.0f}")

Fine-tuned WITH extreme fake context ($30k comps): $9,995


In [11]:
fake_context = "Here are some similar used car listings with their prices:\n\n" \
    "Price: $32,000\nTitle: 2017 Ford Fiesta SE Hatchback\n...\n\n" \
    "Price: $29,500\nTitle: 2017 Ford Fiesta SE Hatchback\n...\n\n" \
    "Price: $31,200\nTitle: 2017 Ford Fiesta SE Hatchback\n..."

result_with_fake_context = pricer.price.remote(test[0].summary, fake_context)
print(f"Fine-tuned WITH extreme fake context: ${result_with_fake_context:,.0f}")

Fine-tuned WITH extreme fake context: $11,975


In [12]:
# Re-run with the real Fiesta comps (new context position: after description, before PREFIX)
result_with_rag_v2 = pricer.price.remote(test[0].summary, context)
print(f"Fine-tuned WITH real RAG (context after description): ${result_with_rag_v2:,.0f}")

# Re-run with extreme fake $30k comps
result_with_fake_context_v2 = pricer.price.remote(test[0].summary, fake_context)
print(f"Fine-tuned WITH extreme fake context (context after description): ${result_with_fake_context_v2:,.0f}")

Fine-tuned WITH real RAG (context after description): $9,500
Fine-tuned WITH extreme fake context (context after description): $12,975


In [13]:
from tqdm import tqdm

sample_size = 15
sample = test[:sample_size]

results_no_rag = []
results_with_rag = []

for car in tqdm(sample):
    docs, prices = find_similars(car)
    car_context = make_context(docs, prices)

    pred_no_rag = pricer.price.remote(car.summary)
    pred_with_rag = pricer.price.remote(car.summary, car_context)

    results_no_rag.append(pred_no_rag)
    results_with_rag.append(pred_with_rag)

errors_no_rag = [abs(p - c.price) for p, c in zip(results_no_rag, sample)]
errors_with_rag = [abs(p - c.price) for p, c in zip(results_with_rag, sample)]

print(f"Mean absolute error WITHOUT RAG: ${sum(errors_no_rag)/len(errors_no_rag):,.2f}")
print(f"Mean absolute error WITH RAG:    ${sum(errors_with_rag)/len(errors_with_rag):,.2f}")

for i, c in enumerate(sample):
    print(f"Actual: ${c.price:>8,.0f}  |  No RAG: ${results_no_rag[i]:>8,.0f}  |  With RAG: ${results_with_rag[i]:>8,.0f}")

100%|██████████| 15/15 [00:33<00:00,  2.26s/it]

Mean absolute error WITHOUT RAG: $1,691.80
Mean absolute error WITH RAG:    $2,490.93
Actual: $  10,990  |  No RAG: $   9,995  |  With RAG: $   9,500
Actual: $  22,424  |  No RAG: $  21,500  |  With RAG: $  21,499
Actual: $  26,188  |  No RAG: $  29,995  |  With RAG: $  24,999
Actual: $  12,995  |  No RAG: $  13,995  |  With RAG: $  18,995
Actual: $  14,999  |  No RAG: $  13,995  |  With RAG: $  13,995
Actual: $  39,995  |  No RAG: $  37,995  |  With RAG: $  34,995
Actual: $  36,105  |  No RAG: $  40,984  |  With RAG: $  37,995
Actual: $   5,900  |  No RAG: $   6,995  |  With RAG: $   7,995
Actual: $  16,400  |  No RAG: $  18,995  |  With RAG: $  18,991
Actual: $  17,992  |  No RAG: $  18,900  |  With RAG: $  13,995
Actual: $  21,999  |  No RAG: $  22,995  |  With RAG: $  21,500
Actual: $  19,500  |  No RAG: $  18,995  |  With RAG: $  18,995
Actual: $  38,945  |  No RAG: $  37,995  |  With RAG: $  34,995
Actual: $  17,000  |  No RAG: $  17,495  |  With RAG: $  13,991
Actual: $  26,771 

In [14]:
class BaseAgent:
    def price(self, description: str) -> float:
        raise NotImplementedError


class SpecialistAgent(BaseAgent):
    """Wraps the Modal-deployed fine-tuned Llama model. No RAG — proven to be
    the more reliable mode from tonight's testing."""
    def __init__(self):
        import modal
        AutoPricer = modal.Cls.from_name("auto-pricer-service", "AutoPricer")
        self.pricer = AutoPricer()

    def price(self, description: str) -> float:
        return self.pricer.price.remote(description)


class FrontierAgent(BaseAgent):
    """Wraps Gemini 2.5 Flash with RAG comps injection."""
    def __init__(self):
        pass  # find_similars/make_context/litellm already available in notebook scope

    def price(self, description: str) -> float:
        # reuse find_similars/make_context on a lightweight object with .summary
        class _Tmp:
            pass
        car = _Tmp()
        car.summary = description
        docs, prices = find_similars(car)
        context = make_context(docs, prices)
        message = f"""Estimate the price of this used car. Respond with the price only, no explanation.

Car to price:
{description}

{context}"""
        response = completion(model="gemini/gemini-2.5-flash", messages=[{"role": "user", "content": message}])
        text = response.choices[0].message.content
        import re
        match = re.search(r"[-+]?\d*\.\d+|\d+", text.replace(",", ""))
        return float(match.group()) if match else 0.0


class EnsembleAgent(BaseAgent):
    """Averages SpecialistAgent (fine-tuned, no RAG) and FrontierAgent (Gemini+RAG)."""
    def __init__(self, specialist_weight: float = 0.5):
        self.specialist = SpecialistAgent()
        self.frontier = FrontierAgent()
        self.specialist_weight = specialist_weight

    def price(self, description: str) -> float:
        p1 = self.specialist.price(description)
        p2 = self.frontier.price(description)
        return self.specialist_weight * p1 + (1 - self.specialist_weight) * p2

In [15]:
specialist = SpecialistAgent()
frontier = FrontierAgent()
ensemble = EnsembleAgent(specialist_weight=0.5)

results_specialist = []
results_frontier = []
results_ensemble = []

for car in tqdm(sample):
    p_spec = specialist.price(car.summary)
    p_front = frontier.price(car.summary)
    p_ens = 0.5 * p_spec + 0.5 * p_front

    results_specialist.append(p_spec)
    results_frontier.append(p_front)
    results_ensemble.append(p_ens)

def mae(preds, sample):
    return sum(abs(p - c.price) for p, c in zip(preds, sample)) / len(preds)

print(f"Specialist alone MAE: ${mae(results_specialist, sample):,.2f}")
print(f"Frontier (Gemini+RAG) alone MAE: ${mae(results_frontier, sample):,.2f}")
print(f"Ensemble (50/50) MAE: ${mae(results_ensemble, sample):,.2f}")

for i, c in enumerate(sample):
    print(f"Actual: ${c.price:>8,.0f}  |  Specialist: ${results_specialist[i]:>8,.0f}  |  Frontier: ${results_frontier[i]:>8,.0f}  |  Ensemble: ${results_ensemble[i]:>8,.0f}")

100%|██████████| 15/15 [04:28<00:00, 17.90s/it]

Specialist alone MAE: $1,691.80
Frontier (Gemini+RAG) alone MAE: $1,599.20
Ensemble (50/50) MAE: $1,202.10
Actual: $  10,990  |  Specialist: $   9,995  |  Frontier: $  10,700  |  Ensemble: $  10,348
Actual: $  22,424  |  Specialist: $  21,500  |  Frontier: $  23,300  |  Ensemble: $  22,400
Actual: $  26,188  |  Specialist: $  29,995  |  Frontier: $  27,900  |  Ensemble: $  28,948
Actual: $  12,995  |  Specialist: $  13,995  |  Frontier: $  11,200  |  Ensemble: $  12,598
Actual: $  14,999  |  Specialist: $  13,995  |  Frontier: $  13,900  |  Ensemble: $  13,948
Actual: $  39,995  |  Specialist: $  37,995  |  Frontier: $  37,750  |  Ensemble: $  37,872
Actual: $  36,105  |  Specialist: $  40,984  |  Frontier: $  37,100  |  Ensemble: $  39,042
Actual: $   5,900  |  Specialist: $   6,995  |  Frontier: $   6,445  |  Ensemble: $   6,720
Actual: $  16,400  |  Specialist: $  18,995  |  Frontier: $  18,198  |  Ensemble: $  18,596
Actual: $  17,992  |  Specialist: $  18,900  |  Frontier: $  22,5

In [16]:
import sys
sys.path.append('..')

from auto_pricer.agents import SpecialistAgent, FrontierAgent, EnsembleAgent

ensemble = EnsembleAgent()
result = ensemble.price(test[0].summary)
print(f"Actual: ${test[0].price:,.0f}  |  Ensemble (from .py files): ${result:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Actual: $10,990  |  Ensemble (from .py files): $10,898


In [17]:
import os
print(f"Current working directory: {os.getcwd()}")
print(f"Collection count (from 7_rag.ipynb's own connection): {collection.count():,}")

Current working directory: d:\mrsha\Projects\AutoPricer\notebooks
Collection count (from 7_rag.ipynb's own connection): 231,947
